# Network Intrusion Detection with Machine Learning
A guided workflow for analysing CICIDS2017 network traffic and comparing binary threat classifiers.

**Dataset:** Download CICIDS2017 from the [official UNB page](https://www.unb.ca/cic/datasets/ids-2017.html) and put its CSV files in `../data/`.

**Related project:** [Keshav-Kathuria/ai-cybersecurity-unit1-mini-project-keshav](https://github.com/Keshav-Kathuria/ai-cybersecurity-unit1-mini-project-keshav). This notebook is a separately organised learning implementation; see the README for attribution and references.


In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib.pyplot as plt
import pandas as pd
from src.data_utils import load_flow_data, clean_flow_data, binary_target
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score


## 1. Load and inspect the dataset


In [ ]:
data = load_flow_data('../data')
data, cleaning_summary = clean_flow_data(data)
print('Shape:', data.shape)
print('Cleaning:', cleaning_summary)
data.head()


## 2. Exploratory data analysis


In [ ]:
label_counts = data['Label'].value_counts()
display(label_counts)
label_counts.plot(kind='bar', figsize=(12, 4), title='Network traffic labels')
plt.ylabel('Flow count')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()


## 3. Prepare features and target
`BENIGN` is mapped to class 0; all attack labels are mapped to class 1. The imputer and scaler are fitted on training data only.


In [ ]:
y = binary_target(data['Label'])
X = data.drop(columns=['Label']).apply(pd.to_numeric, errors='coerce')
X = X.replace([float('inf'), float('-inf')], float('nan')).dropna(axis=1, how='all')
X = X.loc[:, X.nunique(dropna=True) > 1]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print('Train:', X_train.shape, 'Test:', X_test.shape)


## 4. Train and evaluate models


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=12, class_weight='balanced', random_state=42),
    'MLP Neural Network': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, early_stopping=True, random_state=42),
}
preprocessor = ColumnTransformer([('numeric', Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
]), list(X_train.columns))])
scores = []
for name, model in models.items():
    pipeline = Pipeline([('preprocess', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    score = f1_score(y_test, pred, zero_division=0)
    scores.append({'Model': name, 'F1': score})
    print(f'\\n{name} — F1: {score:.4f}')
    print(classification_report(y_test, pred, target_names=['BENIGN', 'ATTACK'], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['BENIGN', 'ATTACK'])
    plt.title(name)
    plt.show()
pd.DataFrame(scores).sort_values('F1', ascending=False)


## 5. Interpretation and limitations
Compare recall and F1, not accuracy alone. CICIDS2017 is imbalanced, and random flow-level splits can overestimate generalisation when similar flows appear in both partitions. Stronger follow-up work could use time-based or group-based splits, per-attack metrics, and feature-ablation tests. This is an educational classifier, not a production intrusion-prevention system.
